<a href="https://colab.research.google.com/github/Squad-Nina-da-Hora/wmc-projeto-final-lifestyle/blob/steph%2Ffeature%2Feda/analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# **Análise de Sono e Estilo de Vida**
---


🎯 **Objetivo:**


---


Projeto Final

Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## 1. Configurações Iniciais

Variáveis da base de dados:

- `Person ID`: Identificador único de cada indivíduo.
- `Gender`: Gênero da pessoa (Masculino/Feminino).
- `Age`: Idade da pessoa em anos.
- `Occupation`: Ocupação ou profissão da pessoa.
- `Sleep Duration` (hours): Número de horas que a pessoa dorme por dia.
- `Quality of Sleep` (scale: 1-10): Avaliação subjetiva da qualidade do sono, variando de 1 a 10.
- `Physical Activity Level` (minutes/day): Número de minutos de atividade física diária.
- `Stress Level` (scale: 1-10): Avaliação subjetiva do nível de estresse, variando de 1 a 10.
- `BMI Category`: Categoria de IMC (por exemplo: Abaixo do peso, Normal, Sobrepeso).
- `Blood Pressure` (systolic/diastolic): Medida da pressão arterial, indicada como pressão sistólica sobre diastólica.
- `Heart Rate` (bpm): Frequência cardíaca em repouso, em batimentos por minuto.
- `Daily Steps`: Número de passos dados por dia.
- `Sleep Disorder`: Presença ou ausência de distúrbio do sono (Nenhum, Insomnia, Sleep Apnea).

In [ ]:
# ==============================
# IMPORTACOES
# ==============================

import kagglehub    # Para baixar datasets do Kaggle
import numpy as np  # Para operacoes numericas e arrays
import pandas as pd  # Para manipulaco e analise de data frames
#from IPython.display import display, Markdown
from scipy import stats # Para o teste de normalidade
# Bibliotecas para criacao de graficos
import matplotlib.pyplot as plt
import seaborn as sns
"""
import missingno as msno
# Bibliotecas para criacao de modelos de ML
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score,
                            recall_score, f1_score, roc_auc_score,
                            classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
"""

# Carregamento da base de dados
arquivo = 'Sleep_health_and_lifestyle_dataset.csv'
url = f'uom190346a/sleep-health-and-lifestyle-dataset'
df = (pd.read_csv(f'{kagglehub.dataset_download(url)}/{arquivo}')
        .set_index('Person ID'))

In [ ]:
# ==============================
# VARIAVEIS PARA REUTILIZACAO
# ==============================

ALVO = 'Quality of Sleep' # Constante da varivel alvo do modelo
ALVO_CAT = 'sono_classificacao' # Contante da variavel alvo categorizada

# Configuracoes visuais dos graficos

PALETA = 'flare'

In [ ]:
# ==============================
# FUNCOES
# ==============================

def print_titulo(txt):
  print(f'{"="*80}\n{txt}\n{"="*80}')

def checar_normalidade(dados):
  """
  Executa o teste de Shapiro-Wilk para avaliar a normalidade das colunas
  numericas de um df e sugere o tipo de correlacao ideal (Pearson ou Spearman).
  """
  resultados = []
  # Filtra apenas as colunas numericas do df fornecido
  colunas_numericas = dados.select_dtypes(include=['number']).columns

  for col in colunas_numericas:
    # Executa o teste de Shapiro-Wilk desconsiderando valores nulos
    shapiro_stat, shapiro_p = stats.shapiro(dados[col].dropna())

    # Criterio: p-valor maior que 0.05 sugere distribuicao normal
    esta_normal = shapiro_p > 0.05
    sugestao_correlacao = 'Pearson' if esta_normal else 'Spearman'

    resultados.append({
      'Variável': col,
      'Shapiro': f'{shapiro_stat:.4f}',
      'p-valor (Shapiro)': f'{shapiro_p:.4e}',
      'Normal?': 'Sim' if esta_normal else 'Não',
      'Recomendação': sugestao_correlacao
    })

  return pd.DataFrame(resultados)

def obter_perfis_dominantes(df, cols_grupo, col_resposta=None, top_n=5):
  """
  Retorna os perfis (combinacoes de cols_grupo) mais frequentes.

  Se col_resposta for None, retorna so a frequencia de cada perfil.
  Caso contrario, mostra a distribuicao da resposta dentro de cada perfil.
  """
  if col_resposta is None:
    perfis_perc = df.groupby(cols_grupo, observed=True).size() / len(df) * 100
    perfis = perfis_perc.sort_values(ascending=False).head(top_n)
    return perfis.to_frame('Total %%')

  ct = pd.crosstab([df[c] for c in cols_grupo], df[col_resposta],
                   normalize='all').mul(100)
  ct['Total %'] = ct.sum(axis=1)
  ct = ct.sort_values('Total %', ascending=False).head(top_n)
  return ct

##  2. Análise exploratória dos dados

In [ ]:
df.info()

In [ ]:
print_titulo('Describe das colunas numericas')
display(df.describe())

print_titulo('Describe das colunas categoricas')
df.describe(include=['category', 'object', 'string'])

In [ ]:
print_titulo('Verificação de dados duplicados')
dados_duplicados = df.index.duplicated().sum()

if dados_duplicados:
  df = df.drop_duplicates()

print(f'\nForam removidas {dados_duplicados} linhas duplicadas.')

In [ ]:
print_titulo('Verificação de valores únicos nas colunas categóricas')
for col in df.select_dtypes(include=['category', 'object', 'string']).columns:
  print(f'\nColuna: {col}')
  print(df[col].unique())

In [ ]:
# Converte colunas categorias para o tipo 'category'
cols_categoricas = ['Gender', 'Occupation', 'Sleep Disorder']
df[cols_categoricas] = df[cols_categoricas].astype('category')

# Padroniza categorias iguais com nomes diferentes em 'BMI Category', converte
# para categorica e define a ordem das categorias
df['BMI Category'] = df['BMI Category'].str.replace('Normal Weight', 'Normal',
                                                    regex=False)
df['BMI Category'] = pd.Categorical(
    df['BMI Category'], categories=['Normal', 'Overweight', 'Obese'],
    ordered=True)

In [ ]:
# Separa e converte a pressao arterial em sistolica e diastolica (int) e
# calcula a PAM
pressoes_separadas = (df['Blood Pressure'].str.split('/', expand=True)
                                          .astype(int))
sistolica = pressoes_separadas[0]
diastolica = pressoes_separadas[1]
df['PAM'] = (sistolica + 2 * diastolica) / 3

In [ ]:
# Cria coluna booleana indicando se a pessoa tem algum disturbio de sono
df['tem_disturbio'] = (df['Sleep Disorder'].isin(df['Sleep Disorder'].cat
                                                 .categories).astype(bool))

# Cria a coluna com 3 classificacoes, espacadas uniformemente entre a idade
# minima e maxima
idade_rotulos = ['Mais jovem', 'Intermediária', 'Mais velha']
idade_bins = np.linspace(df['Age'].min(), df['Age'].max(), 4)

df['idade_classificacao'] = pd.cut(df['Age'], bins=idade_bins,
                                   labels=idade_rotulos, include_lowest=True)

In [ ]:
# Cria a coluna de classificacao da qualidade de sono (var alvo)
sono_rotulos = ['Ruim', 'Moderada', 'Boa']
sono_bins = [0, 4, 6, 10]

df[ALVO_CAT] = pd.cut(df[ALVO], bins=sono_bins, labels=sono_rotulos,
                      include_lowest=True)

In [ ]:
# Analisa distribuicao e desbalanceamento da variavel alvo
n_alvo_cat = df[ALVO_CAT].nunique() # quantidade de categorias na variavel alvo
alvo_cores = sns.color_palette(PALETA, n_colors=n_alvo_cat)

# Tabela
display((df[ALVO_CAT].value_counts(normalize=True) * 100).rename('Percentual'))

# Grafico
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x=ALVO_CAT, palette=alvo_cores, hue=ALVO_CAT,
                   legend=False)
plt.title(f'Distribuição da Variável Alvo `{ALVO_CAT}`', fontsize=12,
          fontweight='bold')
plt.xlabel(f'{ALVO_CAT}')
plt.ylabel('Contagem')

# Adiciona porcentagens nas barras
total = len(df)
for p in ax.patches:
  height = p.get_height()
  ax.annotate(f'{(height/total)*100:.1f}%',
              (p.get_x() + p.get_width() / 2., height), ha='center',
              va='bottom', xytext=(0, 3), textcoords='offset points')
plt.tight_layout()
plt.show()

In [ ]:
# Remove colunas que nao serao utilizadas na analise
df = df.drop(['Blood Pressure'], axis=1)

In [ ]:
print_titulo('Diagnóstico de Tipagem e Qualidade')
nulos_str = (df.isnull().sum().astype(str)
  + ' ('
  + (df.isnull().mean() * 100).map('{:.1f}%'.format)
  + ')')

pd.DataFrame({
  'Tipo': df.dtypes.astype(str),
  'Valores Nulos': nulos_str,
  'Valores Únicos': df.nunique(),
}).sort_values('Valores Nulos', ascending=False)

In [ ]:
print_titulo('Verificação de normalidade das variáveis numéricas')
df_normalidade = checar_normalidade(df)
df_normalidade

In [ ]:
df.head()

### Gráficos

In [ ]:
# Seleciona apenas as variaveis features
features = [col for col in df.columns if col != ALVO and col != ALVO_CAT]

# Seleciona apenas as variaveis features numericas
df_features_num = df[features].select_dtypes(['number'])
# Seleciona apenas as variaveis features categoricas
df_features_cat = df[features].select_dtypes(['category', 'object', 'string'])

# Cores para a variavel alvo nos graficos
alvo_cores = sns.color_palette(PALETA, n_colors=df[ALVO_CAT].nunique())

In [ ]:
# Plota os histogramas das variaveis numericas
fig, axes = plt.subplots(3, 3,figsize=(16, 16))
axes = axes.flatten()
fig.suptitle('Distribuição das Variáveis Numéricas', fontsize=14,
             fontweight='bold', y=1.01)

for i, var in enumerate(df_features_num.columns):
  sns.histplot(data=df, x=var, kde=True, ax=axes[i], color=alvo_cores[2])
  axes[i].set_title(var, fontweight='bold')
  axes[i].set_xlabel('')
  axes[i].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

In [ ]:
# Plota os countplots das variaveis categoricas
fig, axes = plt.subplots(2, 3,figsize=(16, 8))
fig.suptitle('Frequência das Variáveis Categóricas', fontsize=14,
             fontweight='bold', y=1.01)
axes = axes.flatten()

for i, var in enumerate(df_features_cat.columns):
  ordem = df[var].value_counts().index  # ordena por frequencia
  sns.countplot(data=df, y=var, order=ordem, ax=axes[i], hue=var,
                palette=PALETA, legend=False, dodge=False, stat='percent')
  axes[i].set_title(var, fontweight='bold')
  axes[i].set_xlabel('Frequência (%)')
  axes[i].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Correlacao da variavel alvo com as features numericas
df_corr = df.copy()
# Female/Male → 0/1
df_corr['Gender'] = df_corr['Gender'].cat.codes
# Normal/Overwwight/Obese → 0/1/2
df_corr['BMI Category'] = df_corr['BMI Category'].cat.codes
# False/True → 0/1
df_corr['tem_disturbio'] = df_corr['tem_disturbio'].astype('int8')
 # seleciona apenas as numericas
df_corr = df_corr.select_dtypes(include=['number'])

# Calcula e ordena a correlacao
df_corr_alvo = (df_corr.corrwith(df_corr[ALVO], method='spearman')
                       .sort_values(ascending=False).to_frame(name=ALVO))

# Plota heatmap da correlacao
fig, axes = plt.subplots(figsize=(4, 5))
plt.title(f'Correlação com a Variável Alvo\n`{ALVO}`', fontsize=14,
          fontweight='bold')
sns.heatmap(df_corr_alvo, annot=True, fmt='.2f', vmin=-1, vmax=1,
            cmap=sns.diverging_palette(370, 220, as_cmap=True))
plt.tight_layout()
plt.show()

In [ ]:
# Cria ranking da correlacao das variaveis, baseado nos
# valores absolutos >= 0.6, decrescentemente
df_rank = df_corr_alvo.drop(ALVO).rename(columns={ALVO: 'valor'})
df_rank['valor_abs'] = df_rank['valor'].abs()
df_rank = (df_rank.query('valor_abs >= 0.6')
                  .sort_values(by='valor_abs', ascending=False)
                  .reset_index(names='feature'))
df_rank['rank'] = (df_rank['valor_abs'].rank(ascending=False, method='dense')
                                       .astype(int))
df_rank = df_rank.set_index('rank')[['feature', 'valor', 'valor_abs']]

# Plota grafico de barras do ranking
fig, ax = plt.subplots(figsize=(6, 3))
plt.title('Ranking da Correlação das Variáveis', fontsize=14, fontweight='bold')
sns.barplot(x='valor', y=df_rank['feature'], data=df_rank,
            palette=PALETA + '_r', hue=df_rank['feature'], legend=False)

for container in ax.containers:
  ax.bar_label(container, fmt='%.2f', padding=3)

plt.xlabel('')
plt.ylabel('')
plt.margins(x=0.15)
plt.tight_layout()
plt.show()

In [ ]:
# Plota os Boxplots
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f'Boxplots da Variável Alvo `{ALVO_CAT}`', fontsize=14,
             fontweight='bold')
axes = axes.flatten()

for i, var in enumerate(df_features_num.columns):
  sns.boxplot(data=df, x=ALVO_CAT, y=var, ax=axes[i], palette=alvo_cores,
              hue=ALVO_CAT, legend=False)
  axes[i].set_title(var, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Faz o cruzamento de variavel alvo e features-alvo com as features categoricas
# para encontrar perfis dominantes
perfis_features_padrao = ['Occupation', 'BMI Category', 'Sleep Disorder']

# Define as colunas de grupos para cada variavel alvo
perfis_dic = {
  ALVO_CAT: ['Gender'] + perfis_features_padrao + ['idade_classificacao'],
  'Gender': [ALVO_CAT] + perfis_features_padrao + ['idade_classificacao'],
  'idade_classificacao': ['Gender'] + perfis_features_padrao + [ALVO_CAT],
}
perfis = {} # Dicionario para armazenar os resultados
for alvo, features in perfis_dic.items():
  print_titulo(f'Perfis dominantes em `{alvo}`')
  perfis[alvo] = obter_perfis_dominantes(df, features, alvo)
  display(perfis[alvo].round(2))

## 3. Modelo de ML

### 3.1. Avaliação dos Modelos

## 4. Cálculo do gap de Recall entre as classes

## 5. Validação cruzada e generalização

## 6. Conclusão Geral